# 04 — Test Module 2: RFF + STE K-means Clustering

Verify that the differentiable clustering produces sensible clusters and gradients flow through STE.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
PROJECT_ROOT = '/content/drive/MyDrive/Project_GraphML/ms-zerogad'
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from ms_zerogad.data.loader import load_graph_dataset
from ms_zerogad.data.preprocessing import sparse_to_torch_dense, feature_to_torch
from ms_zerogad.modules.unification import GlobalUnification
from ms_zerogad.modules.clustering import RFFClustering

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## Load Cora and unify

In [ ]:
A_sp, X_sp, y_np = load_graph_dataset('/content/drive/MyDrive/Project_GraphML/ms-zerogad/ms_zerogad/data/raw/Cora.mat')
A = sparse_to_torch_dense(A_sp).to(device)
X = feature_to_torch(X_sp, dense=True).to(device)

module1 = GlobalUnification(d_prime=8).to(device)
X_unified = module1(X, A).requires_grad_(True)  # enable grad to test STE

n = X_unified.shape[0]
print(f'Cora: n={n}, d_prime={X_unified.shape[1]}')

## Cluster into n/2 super-nodes

In [ ]:
clustering = RFFClustering(
    k_smoothing=1, sigma=1.0, D_rff=50, d_svd=32, tau=0.5, kmeans_max_iter=20,
).to(device)

import time
t0 = time.time()
P = clustering(X_unified, A, num_clusters=n // 2)
print(f'Clustering time: {time.time() - t0:.2f}s')
print(f'P shape: {P.shape}')

## Check that P is one-hot in forward

In [ ]:
P_detached = P.detach()
row_sums = P_detached.sum(dim=-1)
row_max = P_detached.max(dim=-1).values

print(f'Row sums (should be ~1): min={row_sums.min().item():.4f}, max={row_sums.max().item():.4f}')
print(f'Row max (should be 1): min={row_max.min().item():.4f}, max={row_max.max().item():.4f}')
is_one_hot = (P_detached == 0).logical_or(P_detached == 1).all().item()
print(f'Forward is one-hot: {is_one_hot}')

## Cluster size distribution

In [ ]:
cluster_sizes = P_detached.sum(dim=0).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(cluster_sizes, bins=30, color='steelblue', alpha=0.7)
ax.set_xlabel('Cluster size')
ax.set_ylabel('Count')
ax.set_title(f'Cluster size distribution ({len(cluster_sizes)} clusters)')
plt.show()

print(f'Cluster sizes: min={int(cluster_sizes.min())}, max={int(cluster_sizes.max())}, '
      f'mean={cluster_sizes.mean():.2f}, std={cluster_sizes.std():.2f}')
print(f'Empty clusters: {int((cluster_sizes == 0).sum())}')

## Verify STE: gradient flows through P

In [ ]:
# Construct a dummy loss that depends on P
loss = (P ** 2).sum()  # sum of squared assignments
loss.backward()

# X_unified should have a gradient because of STE
if X_unified.grad is not None and X_unified.grad.abs().sum() > 0:
    print(f'X_unified.grad nonzero: True')
    print(f'  abs sum: {X_unified.grad.abs().sum().item():.4f}')
    print(f'  max:     {X_unified.grad.abs().max().item():.6f}')
else:
    print('WARNING: no gradient flowed through STE!')